In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from stargazer.stargazer import Stargazer
from statsmodels.stats.outliers_influence import variance_inflation_factor as vif

# Смещение из-за пропущенной переменной

## Первый датасет

In [ ]:
df0 = pd.read_excel("../../data1/cps12.xlsx")
df0.head()

Оценим для начала парную модель.

In [ ]:
model0 = sm.OLS.from_formula("ahe ~ age", data=df0).fit()
print(model0.summary())

Добавим переменные `female` и `bachelor`

In [ ]:
model1 = sm.OLS.from_formula("ahe ~ age + female + bachelor", data=df0).fit()
print(model1.summary())

Сравним новую оценку с оценкой парной модели.

In [ ]:
model1.t_test(f"age = {model0.params['age']}")

Статистически значимой разницы нет.

Посмотрим на F-тест для гипотезы ${bachelor}=0, {female}=0$.
Проведем этот тест тремя способами.

In [ ]:
model1.f_test("bachelor = 0, female = 0")

In [ ]:
R = np.zeros((2, 4))
R[0, 2] = R[1, 3] = 1

model1.f_test(R)

In [ ]:
r = np.zeros(2)

model1.f_test((R, r))

Итого, смещения, по всей видимости, нет.

## Второй датасет

In [ ]:
df1 = pd.read_excel("../../data1/CollegeDistance.xls")
df1.head()

Оценим парную модель.

In [ ]:
model0 = sm.OLS.from_formula("ed ~ dist", data=df1).fit()
print(model0.summary())

Добавим дополнительные переменные, оценим несколько спецификаций.

In [ ]:
model1 = sm.OLS.from_formula("ed ~ dist + female + black + hispanic", data=df1).fit()
print(model1.summary())

In [ ]:
model2 = sm.OLS.from_formula(
    "ed ~ dist + female + black + hispanic + momcoll + dadcoll", data=df1
).fit()
print(model2.summary())

In [ ]:
model3 = sm.OLS.from_formula(
    "ed ~ dist + female + black + hispanic + momcoll + dadcoll + cue80",
    data=df1,
).fit()
print(model3.summary())

In [ ]:
model4 = sm.OLS.from_formula(
    "ed ~ dist + female + black + hispanic + momcoll + dadcoll + cue80 + stwmfg80 + incomehi + ownhome + bytest",
    data=df1,
).fit()
print(model4.summary())

Сравним оценку коэффициента при `dist` в последней и парной модели.

Так как изменение получилось статистически значимым, то в изначальной модели, скорее всего, имело место смещение.

In [ ]:
model4.t_test(f"dist = {model0.params['dist']}")

Выведем результаты оценивания всех пяти спецификаций в одной таблице.

In [ ]:
Stargazer([model0, model1, model2, model3, model4])

In [ ]:
print(Stargazer([model0, model1, model2, model3, model4]).render_html())

Рассмотрим критерии качества оцененных моделей.

In [ ]:
res = np.full((5, 4), fill_value=np.nan)

for i, m in enumerate([model0, model1, model2, model3, model4]):
    res[i, 0] = m.rsquared
    res[i, 1] = m.rsquared_adj
    res[i, 2] = m.aic
    res[i, 3] = m.bic

pd.DataFrame(res, columns=np.array(["R2", "R2adj", "AIC", "BIC"]))

## Третий датасет

# Инструментальные переменные

```python
class statsmodels.sandbox.regression.gmm.IV2SLS(
  endog, 
  exog, 
  instrument=None
)
```

Все переменные `exog` инструментируются на все переменные в массиве `instrument`.
Если какие-то переменные из `exog` не должны быть инструментированы, то они должны быть включены в `instrument`.

Пусть мы оцениваем регрессию `ed` на `dist` и `bytest`.
Предположим, что у нас есть подозрения на эндогенность `bytest`; предполагаем, что инструментами для `bytest` будут `dadcoll`, `momcoll`, `black` и `hispanic`.

In [ ]:
import statsmodels.sandbox.regression.gmm as sandbox

y = df1.ed
X_iv = sm.add_constant(df1[["dist", "bytest"]])
Instr = df1[
    ["dist", "dadcoll", "momcoll", "black", "hispanic"]
]  # Мы не инструментируем dist, поэтому добавляем его в Instr!

model_iv = sandbox.IV2SLS(y, X_iv, Instr)
model_iv_est = model_iv.fit()
print(model_iv_est.summary())

Тест на слабые инструменты и тест на сверхидентифицирующие ограничения придется проводить вручную.

**Так как модуль, где располагается этот класс называется `sandbox`, я подозреваю, что использовать его опасно!**

## Пакет `linearmodels`

А можно воспользоваться пакетом `linearmodels`, который расширяет пакет `statsmodels`, и в котором уже есть все необходимое.

```python
class linearmodels.iv.model.IV2SLS(
  dependent: IVData | ndarray | DataArray | DataFrame | Series, 
  exog: IVData | ndarray | DataArray | DataFrame | Series | None, 
  endog: IVData | ndarray | DataArray | DataFrame | Series | None, 
  instruments: IVData | ndarray | DataArray | DataFrame | Series | None, 
  *, 
  weights: IVData | ndarray | DataArray | DataFrame | Series | None = None
)
```

In [ ]:
from linearmodels.iv import IV2SLS

X_endog = df1.bytest  # Эндогенные переменные
X_exog = sm.add_constant(df1.dist)  # Экзогенные переменные и константа
X_instr = df1[["dadcoll", "momcoll", "black", "hispanic"]]  # Инструменты

model_iv_alt = IV2SLS(y, X_exog, X_endog, X_instr).fit()
print(model_iv_alt.summary)

Объект с оцененной моделью не содержит методов для t-теста, но его можно эмулировать, взяв корень из соответствующей статистики Вальда. Единственная проблема в том, что знак статистики нужно будет считать руками
$$sgn = \begin{cases}-1&\sqrt{W}<\text{тестируемое значение в }H_0\\1&\text{иначе}\end{cases}$$
Для t-теста на значимость знак t-статистики совпадает со знаком коэффициента.

Ключевое отличие метода `wald_test` в `linearmodels` в том, что метод принимает на вход:
* или матрицу ограничений $R$,
* или матрицу ограничений $R$ и вектор правых частей $r$,
* или строковое представление, но **ОБЯЗАТЕЛЬНО** через именованный параметр `formula`. В `statsmodels` все можно было передавать через один параметр.

Следующие варианты равноправны.

In [ ]:
R = np.array([[0, 1, 0]])
(model_iv_alt.wald_test(R).stat) ** (0.5)

In [ ]:
R = np.array([[0, 1, 0]])
q = np.array([0])
(model_iv_alt.wald_test(R, q).stat) ** (0.5)

In [ ]:
(model_iv_alt.wald_test(formula="dist = 0").stat) ** (0.5)

Для тестирования множественных гипотез просто проводим тест Вальда.

In [ ]:
R = np.array([[0, 1, 0], [0, 0, 1]])
model_iv_alt.wald_test(R)

In [ ]:
model_iv_alt.wald_test(formula="dist=0, bytest=0")

Протестируем на равенство произвольным числам.

In [ ]:
R = np.array([[0, 1, 0], [0, 0, 1]])
q = np.array([1, 2])
model_iv_alt.wald_test(R, q)

In [ ]:
model_iv_alt.wald_test(formula="dist=1, bytest=2")

Диагностика результатов регрессии первого шага.
`f.stat` --- это F-статистика в регрессии эндогенной переменной на инструменты при условии, что и эндогенная переменная, и инструменты ортогонализованы относительно экзогенных переменных.

In [ ]:
model_iv_alt.first_stage.diagnostics

Эта статистика численно равна F-статистике в обычном тесте на слабые инструменты.
Следующий код это показывает:
1. `model_iv_alt.first_stage.individual` содержит результаты оценивания регрессии первого шага в виде словаря, гда ключ --- эндогенная переменная, а значения --- объекты с оцененной моделью.
   Несмотря на то, что пакет `linearmodels` рассматривается как расширение `statsmodels`, интерфейс несколько отличается;
   например, при вызове объекта с оцененной моделью выдается табличка с `summary`, а не унылая строка с названием класса и адресом в памяти.
2. в `wald_test` передана линейная гипотеза о равенстве 0 коэффициентов при инструментах.

In [ ]:
model_iv_alt.first_stage.individual["bytest"]

In [ ]:
model_iv_alt.first_stage.individual["bytest"].wald_test(
    formula="dadcoll=momcoll=black=hispanic=0"
)

Тест Саргана.

In [ ]:
model_iv_alt.sargan

## GMM

Ту же самую модель с инструментальными переменными можнео оценить при помощи Обобщенного метода моментов(GMM).

```python
class linearmodels.iv.model.IVGMM(
  dependent: IVData | ndarray | DataArray | DataFrame | Series, 
  exog: IVData | ndarray | DataArray | DataFrame | Series | None, 
  endog: IVData | ndarray | DataArray | DataFrame | Series | None, 
  instruments: IVData | ndarray | DataArray | DataFrame | Series | None, 
  *, 
  weights: IVData | ndarray | DataArray | DataFrame | Series | None = None, 
  weight_type: str = 'robust', 
  **weight_config: Any
)
```
где `dependent`, `exog`, `endog`, `instruments` имеют очевидное определение,
* `weights` --- веса для наблюдений,
* `weight_type` --- тип весовой функции
  * "unadjusted", "homoskedastic" -- Assumes moment conditions are homoskedastic,
  * "robust", "heteroskedastic" -- Allows for heteroskedasticity by not autocorrelation
  * "kernel" -- Allows for heteroskedasticity and autocorrelation
  * "clustered" -- Allows for one-way cluster dependence. Требует параметра `clusters`.

In [ ]:
from linearmodels.iv import IVGMM

X_endog = df1.bytest  # Эндогенные переменные
X_exog = sm.add_constant(df1.dist)  # Экзогенные переменные и константа
X_instr = df1[["dadcoll", "momcoll", "black", "hispanic"]]  # Инструменты

model_gmm = IVGMM(y, X_exog, X_endog, X_instr, weight_type="unadjusted").fit()
print(model_gmm.summary)

По тестам все аналогично 2SLS.